<a href="https://colab.research.google.com/github/melihdural/Gazi_Bilisim_ML/blob/v5.0/GAZ%C4%B0_ML16_v5_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tabular ML Benchmark: Boosting vs TabNet (Compute-Aware Evaluation)

🎯 Projenin Amacı

Bu proje, tabular veri üzerinde farklı model ailelerinin performansını karşılaştırmak amacıyla geliştirilmiştir.

Özellikle şu soruya cevap aranır:

“Kısıtlı ve gerçekçi compute koşullarında, TabNet gibi derin öğrenme tabanlı modeller gerçekten boosting modellerle rekabet edebilir mi?”

Bu bağlamda iki farklı değerlendirme yaklaşımı uygulanmıştır:
	•	Fair Track → Tüm modellere aynı veri temsili (preprocessing)
	•	Real Track → Her model kendi doğal veri işleme yaklaşımıyla çalışır

⸻

❓ Neden Bu Çalışma Yapıldı?

Tabular veri problemlerinde:
	•	XGBoost, LightGBM, CatBoost gibi boosting modeller uzun süredir dominant
	•	TabNet gibi deep learning modeller ise alternatif olarak öneriliyor

Ancak literatürde ve pratikte şu sorun var:
	•	Karşılaştırmalar çoğu zaman:
	•	ya adil değil (farklı preprocessing)
	•	ya da gerçekçi değil (aşırı tuning / sınırsız compute)

Bu proje:

👉 Adil + gerçekçi + compute-aware bir benchmark kurmayı hedefler

⸻

🧪 Deney Tasarımı

1. Veri Setleri

Benchmark, OpenML üzerinden seçilen farklı tabular veri setleri ile yapılmıştır:

## Model - Dataset Performans Beklentileri

| ID    | Dataset      | Problem Türü / Veri İçeriği | Boyut (Satır x Sütun) | Önerilen Model Tipi | Açıklama |
|------|-------------|-----------------------------|----------------------|---------------------|----------|
| 150  | Covertype   | Coğrafi ve çevresel özelliklerden orman örtüsü tipi sınıflandırma (toprak tipi, yükseklik, eğim, gölge vb.) | ~581K x 54 | XGBoost / LightGBM / CatBoost | Tamamen numeric, yüksek boyutlu ve büyük veri. Sınıflar arası overlap var → kompleks decision boundary gerekir. Boosting modeller üstün. |
| 1590 | Adult       | Bireyin yıllık gelirinin 50K$ üstü/altı tahmini (yaş, eğitim seviyesi, meslek, çalışma saati, medeni durum vb.) | ~48K x 14 | CatBoost | Yoğun kategorik veri (occupation, education, marital-status). Encoding kritik. CatBoost native categorical handling ile avantajlı. |
| 23512| Higgs       | Higgs bozonu tespiti (parçacık çarpışmalarından elde edilen fiziksel ölçümlerle signal/background ayrımı) | ~1M x 28 | XGBoost / LightGBM | Çok büyük ve dense numeric veri. Lineer olmayan kompleks pattern. GPU boosting en iyi performansı verir. |
| 40701| Churn | Müşterinin hizmeti bırakıp bırakmayacağını tahmin (abonelik süresi, kullanım, ödeme davranışı vb.) | ~5K x 20 | Tüm modeller (özellikle boosting) | Küçük veri. Gürültü ve imbalance olabilir. Overfitting riski yüksek → regularization önemli. |
| 40498| Wine quality white | Beyaz şarapların kimyasal özelliklerinden şarap kalitesini (0–10 arası bir puan) tahmin etmeye yönelik | ~5K x 11 | Tüm modeller (özellikle boosting) | Küçük veri. Gürültü ve imbalance olabilir. Overfitting riski yüksek. |
| 40981| Australian  | Kredi başvurusunun onay/red tahmini (finansal durum, kredi geçmişi, demografik özellikler) | 690 x 15 | Tree-based modeller | Küçük ve karışık yapı (numeric + categorical). Veri noisy olabilir. Basit ama stabil modeller avantajlı. |

Bu veri setleri:
	•	farklı boyut
	•	farklı dağılım
	•	farklı feature tipleri
içerecek şekilde seçilmiştir.

⸻

2. Train/Test Ayrımı

Her dataset için:
	•	%70 → train
	•	%10 → validation
	•	%20 → test

⸻

3. Cross Validation

Model değerlendirme:
	•	Stratified K-Fold
	•	Fold sayısı:
	•	dataset büyüklüğüne göre dinamik
	•	Seed:
	•	çoklu seed ile stabilite ölçümü (3 Seed)

⸻

⚖️ Track Yapısı

🔹 Fair Track
Amaç: Modelleri tamamen eşit koşullarda karşılaştırmak
Özellikler:
	•	Ortak preprocessing
	•	OneHotEncoder (feature explosion kontrolü ile)
	•	Aynı input representation

🔹 Real Track
Amaç: Modelleri gerçek kullanım senaryosuna yakın değerlendirmek
Özellikler:
	•	Model-specific preprocessing
	•	TabNet: native categorical encoding, embedding kullanımı
	•	Boosting: klasik veya native preprocessing (ör. CatBoost)

⸻

⏱️ Compute-Aware Benchmark ve Kısıtlar

Bu benchmark’ın en kritik özelliği: Sınırsız compute yok.

Her model için:
	•	zaman limiti (Track ve Hız sınıfına göre, ör. fast modeller için Fair: 7.5 dk, Real: 20 dk)
	•	fold sayısı
	•	seed sayısı
kontrol edilir.

Model hız sınıfları:
```python
MODEL_SPEED_MAP = {
    "LightGBM": "fast",
    "XGBoost": "medium",
    "CatBoost": "medium",
    "TabNet_Fast": "slow",
    "TabNet_Max": "slow",
    "GBT_M": "fast"  # HistGradientBoosting
}
```

Budget Ayarları (Fold ve Seed):
```python
BUDGET_CONFIG = {
    "fair": {"folds": 3, "seeds": 1},
    "real": {
        "fast": {"folds": 5, "seeds": 2},
        "medium": {"folds": 3, "seeds": 2},
        "slow": {"folds": 2, "seeds": 1},
    }
}
```

⸻

🤖 Kullanılan Modeller

Boosting:
	•	XGBoost
	•	LightGBM
	•	CatBoost
	•	HistGradientBoosting (GBT_M)

Deep Learning:
	•	TabNet (Fast)
	•	TabNet (Max)

⸻

💾 Checkpoint ve Google Drive Sistemi

Uzun süren eğitim süreçleri göz önüne alınarak projeye Checkpoint / Resume özelliği eklenmiştir.
	•	Model veya fold sırasında Colab koparsa kaldığı yerden devam edebilir.
	•	Tüm ara sonuçlar Google Drive üzerindeki `Gazi_ML16_Checkpoints` klasörüne yedeklenir.

⸻

📈 Değerlendirme Metrikleri

Ana metrik: `Macro F1 Score`

⸻

🧠 XAI (Explainability)

Benchmark sonrası:
	•	En iyi modeller için SHAP (TreeExplainer) analizi uygulanır.
	•	Sadece Post-hoc feature importance analiz edilir.

⸻

📊 Çıktılar

Benchmark sonunda aşağıdaki dosyalar oluşturulur:
```text
Gazi_ML16_Tum_Sonuclar_All_Tracks.csv
Gazi_ML16_En_Iyi_Modeller_All_Tracks.csv
Gazi_ML16_Ham_Sonuclar_All_Tracks.csv
Gazi_ML16_Fair_Real_Karsilastirma.csv
```

⸻

📌 Sonuçların Doğru Yorumlanması

Bu benchmark şunu ölçer: “Sınırlı compute altında hangi model daha iyi?”
Şunu ölçmez: “Sınırsız tuning ile en iyi model hangisi?”

# Bölüm 1 : Kütüphane Kurulumları

In [ ]:
# =========================
# PACKAGE INSTALL
# =========================

!pip install -q --upgrade \
catboost \
lightgbm \
xgboost \
pytorch-tabnet \
scikit-learn \
"pandas==2.2.2" \
numpy \
matplotlib \
seaborn \
shap

# Bölüm 2 : İçe Aktarmalar ve Sabit Tanımlamaları

In [ ]:
# =========================
# IMPORTS
# =========================

import pandas as pd
import numpy as np
import time
import random
import pickle
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import torch
import os

from datetime import datetime

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.metrics import confusion_matrix

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder
)

from sklearn.model_selection import train_test_split, StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    log_loss
)

from sklearn.datasets import fetch_openml

from sklearn.ensemble import HistGradientBoostingClassifier

from pytorch_tabnet.tab_model import TabNetClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from google.colab import (sheets, drive)
from IPython.display import display
from scipy.stats import ttest_rel

warnings.filterwarnings("ignore")

drive.mount("/content/drive")


DRIVE_SNAPSHOT_DIR = "/content/drive/MyDrive/Gazi_ML16_Checkpoints"
os.makedirs(DRIVE_SNAPSHOT_DIR, exist_ok=True)


# =========================
# TRACK & BUDGET CONFIG
# =========================

TRACK_MODE = "real"  # "fair" veya "real"

MODEL_SPEED_MAP = {
    "XGBoost": "medium",
    "LightGBM": "fast",
    "CatBoost": "medium",
    "TabNet": "slow",
    "GBT_M": "fast"
}

BUDGET_CONFIG = {
    "fair": {"folds": 3, "seeds": 1},
    "real": {
        "fast": {"folds": 5, "seeds": 2},
        "medium": {"folds": 3, "seeds": 2},
        "slow": {"folds": 2, "seeds": 1},
    }
}

TABNET_DATASET_OVERRIDES = {
    150: {
        "fair": {
            "TabNet": {"folds": 2, "seeds": 1, "max_epochs": 15},
        },
        "real": {
            "TabNet": {"folds": 2, "seeds": 1, "max_epochs": 25},
        }
    }
}

MODEL_TIME_LIMIT_SECONDS = {
    "fair": {
        "fast": 450,     # 7.5 dk
        "medium": 900,   # 15 dk
        "slow": 1200,    # 20 dk
    },
    "real": {
        "fast": 1200,    # 20 dk
        "medium": 1800,  # 30 dk
        "slow": 2400,    # 40 dk
    }
}

# =========================
# EXPERIMENT SETTINGS
# =========================

SEEDS = [42, 2024, 1337]

OPENML_DATASET_IDS = [150, 1590, 23512, 40701, 40498, 40981]

PRIMARY_METRIC = "Macro_F1"

OUTER_TEST_SIZE = 0.20

TABNET_EPOCHS = 50
BOOSTING_ROUNDS = 300


# =========================
# GLOBAL REPRODUCIBILITY
# =========================

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)

    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


# =========================
# GLOBAL STORAGE
# =========================

results_list = []
trained_models = {}


# Bölüm 3 : Veri Çekme, Bölme ve EDA Fonksiyonları

In [ ]:
# =========================
# DATA LOADING & PREP
# =========================

def load_openml_data(dataset_id):
    print(f"--- Veri Seti Yükleniyor: OpenML ID {dataset_id} ---")

    data = fetch_openml(data_id=dataset_id, as_frame=True, parser="auto")
    df = data.frame.copy()

    target_name = data.target.name if hasattr(data.target, "name") else df.columns[-1]

    X = df.drop(columns=[target_name]).copy()
    y = df[target_name].copy()

    metadata = {
        "dataset_id": dataset_id,
        "dataset_name": data.details.get("name", f"DATASET_{dataset_id}").upper(),
    }

    return X, y, metadata


# =========================
# TARGET PREP
# =========================

def prepare_target(y):
    y_processed = y.copy()

    if y_processed.dtype == "object" or str(y_processed.dtype).startswith("category"):
        y_processed = LabelEncoder().fit_transform(y_processed)

    return pd.Series(y_processed).reset_index(drop=True)


# =========================
# SPLIT (SIMPLIFIED)
# =========================

def split_data(X, y, seed):

    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    # %20 test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=seed,
        stratify=y
    )

    # kalan %80 → %70 train / %10 val
    val_ratio = 0.125  # 0.10 / 0.80

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=val_ratio,
        random_state=seed,
        stratify=y_temp
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


# =========================
# EDA (OPTIONAL)
# =========================

def perform_eda(X, y_target, dataset_id, dataset_name):

    # 🔥 compute budget için kapatılabilir
    RUN_EDA = False

    if not RUN_EDA:
        return

    print(f"\n--- OpenML ID {dataset_id} için EDA ---")

    if len(X) > 3000:
        sample_idx = np.random.choice(len(X), 3000, replace=False)
        X = X.iloc[sample_idx]
        y_target = y_target.iloc[sample_idx]

    # sadece numeric kolonlar
    num_cols = X.select_dtypes(include=[np.number]).columns

    if len(num_cols) > 0:
        plt.figure(figsize=(8, 6))
        sns.heatmap(X[num_cols].corr(), cmap="coolwarm")
        plt.title(f"{dataset_name} - Correlation")
        plt.show()


# =========================
# MODEL VISUALIZATION (OPTIONAL)
# =========================

def perform_model_base_plot(*args, **kwargs):
    # 🔥 benchmark sırasında kapalı
    return

# Bölüm 4 : Budget Helper

In [ ]:
# =========================
# BUDGET HELPER
# =========================

def get_model_budget(model_name):
    """
    Modele göre kaç fold ve kaç seed kullanılacağını belirler
    """

    if TRACK_MODE == "fair":
        return BUDGET_CONFIG["fair"]["folds"], BUDGET_CONFIG["fair"]["seeds"]

    speed = MODEL_SPEED_MAP.get(model_name, "medium")

    config = BUDGET_CONFIG["real"][speed]

    return config["folds"], config["seeds"]


def get_tabnet_override(dataset_id, track_mode, model_name):
    return (
        TABNET_DATASET_OVERRIDES
        .get(dataset_id, {})
        .get(track_mode, {})
        .get(model_name, None)
    )


def get_model_time_limit(track_mode, model_name):
    speed = MODEL_SPEED_MAP.get(model_name, "medium")
    return MODEL_TIME_LIMIT_SECONDS[track_mode][speed]


# Bölüm 5 : Ön İşleme

In [ ]:
# =========================
# PREPROCESSOR (TRACK AWARE)
# =========================

def build_preprocessor(X, model_name=None):

    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

    # ---------- NUMERIC ----------
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    # ---------- CATEGORICAL ----------
    if TRACK_MODE == "fair":

        # fair: herkes aynı temsil
        categorical_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                max_categories=50
            )),
        ])

    else:
        # real: sadece klasik modellere preprocessing
        categorical_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )),
        ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

# Bölüm 6 : Model Tanımlamaları

In [ ]:
def get_model_builders(seed, n_classes=2):

    catboost_loss = "Logloss" if n_classes == 2 else "MultiClass"
    xgb_metric = "logloss" if n_classes == 2 else "mlogloss"
    lgbm_objective = "binary" if n_classes == 2 else "multiclass"

    return {

        # -------------------------
        # XGBOOST
        # -------------------------
        "XGBoost": lambda: XGBClassifier(
            random_state=seed,
            eval_metric=xgb_metric,
            n_jobs=-1,
            tree_method="hist",
            device="cuda",
            max_depth=6,
            n_estimators=BOOSTING_ROUNDS,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
        ),

        # -------------------------
        # LIGHTGBM
        # -------------------------
        "LightGBM": lambda: LGBMClassifier(
            random_state=seed,
            device="gpu",
            n_jobs=-1,
            objective=lgbm_objective,
            n_estimators=BOOSTING_ROUNDS,
            learning_rate=0.1,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            verbosity=-1,
            min_child_samples=20,
        ),

        # -------------------------
        # CATBOOST (SAFE GPU/CPU)
        # -------------------------
        "CatBoost": lambda: CatBoostClassifier(
            random_state=seed,
            verbose=0,
            task_type="GPU" if torch.cuda.is_available() else "CPU",
            **({"devices": "0"} if torch.cuda.is_available() else {}),
            iterations=BOOSTING_ROUNDS,
            learning_rate=0.1,
            depth=6,
            loss_function=catboost_loss,
        ),

        # -------------------------
        # TABNET
        # -------------------------
        "TabNet": lambda: TabNetClassifier(
            n_d=16,
            n_a=16,
            n_steps=3,
            gamma=1.3,
            seed=seed,
            verbose=0,
            device_name="cuda",
            optimizer_params=dict(lr=1e-2),
        ),

        # -------------------------
        # HIST GBT
        # -------------------------
        "GBT_M": lambda: HistGradientBoostingClassifier(
            max_depth=5,
            max_iter=BOOSTING_ROUNDS,
            learning_rate=0.1,
            random_state=seed,
        ),
    }


# Bölüm 7 : Model Fit

In [ ]:
class CatBoostNativePipeline(BaseEstimator, ClassifierMixin):
    def __init__(self, model):
        self.model = model
        self.feature_names_ = None
        self.numeric_features_ = None
        self.cat_features_ = None
        self.numeric_fill_values_ = None
        self.cat_fill_values_ = None

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("CatBoostNativePipeline expects X as a pandas DataFrame.")

        X_fit = X.copy()

        self.feature_names_ = X_fit.columns.tolist()
        self.numeric_features_ = X_fit.select_dtypes(include=[np.number]).columns.tolist()
        self.cat_features_ = X_fit.select_dtypes(exclude=[np.number]).columns.tolist()

        if self.numeric_features_:
            X_fit[self.numeric_features_] = X_fit[self.numeric_features_].apply(pd.to_numeric, errors="coerce")
            self.numeric_fill_values_ = X_fit[self.numeric_features_].median()
            X_fit[self.numeric_features_] = X_fit[self.numeric_features_].fillna(self.numeric_fill_values_)
        else:
            self.numeric_fill_values_ = pd.Series(dtype=float)

        self.cat_fill_values_ = {}
        for col in self.cat_features_:
            # KRITIK FIX
            series = X_fit[col].astype(object)
            mode = series.mode(dropna=True)
            fill_value = mode.iloc[0] if not mode.empty else "__missing__"
            self.cat_fill_values_[col] = fill_value
            X_fit[col] = series.fillna(fill_value).astype(str)

        self.model.fit(
            X_fit[self.feature_names_],
            y,
            cat_features=self.cat_features_,
            verbose=False
        )
        return self

    def transform_features(self, X):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("CatBoostNativePipeline expects X as a pandas DataFrame.")

        X_t = X.copy()[self.feature_names_]

        if self.numeric_features_:
            X_t[self.numeric_features_] = X_t[self.numeric_features_].apply(pd.to_numeric, errors="coerce")
            X_t[self.numeric_features_] = X_t[self.numeric_features_].fillna(self.numeric_fill_values_)

        for col in self.cat_features_:
            fill_value = self.cat_fill_values_.get(col, "__missing__")
            # KRITIK FIX
            X_t[col] = X_t[col].astype(object).fillna(fill_value).astype(str)

        return X_t

    def predict(self, X):
        return self.model.predict(self.transform_features(X))

    def predict_proba(self, X):
        return self.model.predict_proba(self.transform_features(X))

    def get_feature_names_out(self):
        return np.array(self.feature_names_)

class TabNetPipeline(BaseEstimator, ClassifierMixin):
    def __init__(self, model, max_epochs=TABNET_EPOCHS):
        self.model = model
        self.max_epochs = max_epochs

        self.feature_names_ = None
        self.num_cols_ = None
        self.cat_cols_ = None
        self.num_fill_values_ = None
        self.cat_mapping_ = None
        self.cat_idxs = None
        self.cat_dims = None

    def _fit_transform_frame(self, X):
        X_df = X.copy()

        self.feature_names_ = X_df.columns.tolist()
        self.num_cols_ = X_df.select_dtypes(include=[np.number]).columns.tolist()
        self.cat_cols_ = X_df.select_dtypes(exclude=[np.number]).columns.tolist()

        # numeric
        if self.num_cols_:
            X_df[self.num_cols_] = X_df[self.num_cols_].apply(pd.to_numeric, errors="coerce")
            self.num_fill_values_ = X_df[self.num_cols_].median()
            X_df[self.num_cols_] = X_df[self.num_cols_].fillna(self.num_fill_values_)
        else:
            self.num_fill_values_ = pd.Series(dtype=float)

        # categorical
        self.cat_mapping_ = {}
        cat_idxs = []
        cat_dims = []

        for i, col in enumerate(self.feature_names_):
            if col in self.cat_cols_:
                series = X_df[col].astype(object).fillna("__missing__").astype(str)

                categories = pd.Index(sorted(series.unique().tolist()))
                mapping = {cat: idx for idx, cat in enumerate(categories)}

                self.cat_mapping_[col] = mapping
                X_df[col] = series.map(mapping).astype(np.int64)

                cat_idxs.append(i)
                cat_dims.append(len(mapping) + 1)  # unknown category için 1 ekstra

        self.cat_idxs = cat_idxs
        self.cat_dims = cat_dims

        return X_df[self.feature_names_].values.astype(np.float32)

    def transform(self, X):
        X_df = X.copy()[self.feature_names_]

        # numeric
        if self.num_cols_:
            X_df[self.num_cols_] = X_df[self.num_cols_].apply(pd.to_numeric, errors="coerce")
            X_df[self.num_cols_] = X_df[self.num_cols_].fillna(self.num_fill_values_)

        # categorical
        for col in self.cat_cols_:
            series = X_df[col].astype(object).fillna("__missing__").astype(str)
            mapping = self.cat_mapping_[col]
            unknown_code = len(mapping)
            X_df[col] = series.map(lambda v: mapping.get(v, unknown_code)).astype(np.int64)

        return X_df[self.feature_names_].values.astype(np.float32)

    def fit(self, X, y, X_val=None, y_val=None):
        X_train_np = self._fit_transform_frame(X)
        y_train_np = np.asarray(y)

        fit_kwargs = dict(
            X_train=X_train_np,
            y_train=y_train_np,
            max_epochs=self.max_epochs,
            patience=3,
            batch_size=4096,
            virtual_batch_size=512,
        )

        if X_val is not None and y_val is not None:
            X_val_np = self.transform(X_val)
            y_val_np = np.asarray(y_val)

            fit_kwargs["eval_set"] = [(X_val_np, y_val_np)]
            fit_kwargs["eval_name"] = ["valid"]
            fit_kwargs["eval_metric"] = ["accuracy"]

        self.model.fit(**fit_kwargs)
        return self

    def predict(self, X):
        return self.model.predict(self.transform(X))

    def predict_proba(self, X):
        return self.model.predict_proba(self.transform(X))


# =========================
# DYNAMIC EPOCH SETTER
# =========================
def get_dynamic_tabnet_epochs(X):
    rows, cols = X.shape
    if rows > 150000:
        return 15  # Çok Büyük
    elif rows > 40000 or cols > 50:
        return 25  # Orta-Büyük veya Yüksek Boyutlu
    else:
        return 50  # Küçük


# =========================
# MODEL FIT
# =========================

def fit_model(model_name, model, X_train, y_train, X_val=None, y_val=None, max_epochs=None):

    # TabNet için dinamik epoch kontrolü
    if "TabNet" in model_name:
        calc_epochs = max_epochs if max_epochs is not None else get_dynamic_tabnet_epochs(X_train)
    else:
        calc_epochs = None

    # FAIR: herkes aynı ortak preprocessing pipeline’dan geçsin
    if TRACK_MODE == "fair":
        preprocessor = build_preprocessor(X_train, model_name=model_name)

        if "TabNet" in model_name:
            # TabNet'in max_epochs limitini FAIR modda da uygulamak için Pipeline içine sarmalıyoruz.
            fitted_model = TabNetPipeline(model=clone(model), max_epochs=calc_epochs)
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("model", fitted_model)
            ])
            pipeline.fit(X_train, y_train)
            return pipeline
        else:
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("model", clone(model))
            ])
            pipeline.fit(X_train, y_train)
            return pipeline

    # REAL: model-native kullanım
    if model_name == "CatBoost":
        fitted_model = CatBoostNativePipeline(model=clone(model))
        fitted_model.fit(X_train, y_train)
        return fitted_model

    if "TabNet" in model_name:
        fitted_model = TabNetPipeline(
            model=clone(model),
            max_epochs=calc_epochs
        )
        fitted_model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
        return fitted_model

    preprocessor = build_preprocessor(X_train, model_name=model_name)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", clone(model))
    ])

    pipeline.fit(X_train, y_train)
    return pipeline


# Bölüm 8 : Snapshot

In [ ]:
# =========================
# BÖLÜM 8 - SAVE / SNAPSHOT
# =========================

import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

def ensure_snapshot_dir(snapshot_dir=DRIVE_SNAPSHOT_DIR):
    os.makedirs(snapshot_dir, exist_ok=True)


def normalize_checkpoint_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Farklı kolon isimlerini standart hale getirir.
    Standart kolonlar:
      - Dataset
      - Model
      - Seed
      - Track_Mode
      - Status
    """
    if df is None or df.empty:
        return pd.DataFrame()

    df = df.copy()

    rename_map = {}
    for col in df.columns:
        c = str(col).strip().lower()

        if c in ["dataset", "datasetid", "openml_id", "openmlid"]:
            rename_map[col] = "Dataset"
        elif c in ["model", "modelname"]:
            rename_map[col] = "Model"
        elif c in ["seed"]:
            rename_map[col] = "Seed"
        elif c in ["track_mode", "trackmode", "track"]:
            rename_map[col] = "Track_Mode"
        elif c in ["status", "run_status"]:
            rename_map[col] = "Status"

    df = df.rename(columns=rename_map)

    required_cols = ["Dataset", "Model", "Seed", "Track_Mode", "Status"]
    for col in required_cols:
        if col not in df.columns:
            if col == "Status":
                df[col] = "SUCCESS"
            else:
                df[col] = np.nan

    df["Dataset"] = pd.to_numeric(df["Dataset"], errors="coerce")
    df["Seed"] = pd.to_numeric(df["Seed"], errors="coerce")
    df["Model"] = df["Model"].astype(str)
    df["Track_Mode"] = df["Track_Mode"].astype(str)
    df["Status"] = df["Status"].astype(str).str.upper()

    return df


def save_track_snapshot(track_mode, results_df, trained_models=None, snapshot_dir=DRIVE_SNAPSHOT_DIR):
    ensure_snapshot_dir(snapshot_dir)

    if results_df is None or results_df.empty:
        print(f"[SNAPSHOT] Kaydedilecek sonuç yok | Track={track_mode}")
        return

    results_df = normalize_checkpoint_columns(results_df)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    results_path = os.path.join(snapshot_dir, f"{track_mode}_results_{timestamp}.csv")
    latest_results_path = os.path.join(snapshot_dir, f"{track_mode}_results_latest.csv")

    results_df.to_csv(results_path, index=False)
    results_df.to_csv(latest_results_path, index=False)

    meta = {
        "track_mode": track_mode,
        "timestamp": timestamp,
        "row_count": len(results_df),
        "model_object_count": len(trained_models) if trained_models is not None else 0,
    }

    meta_path = os.path.join(snapshot_dir, f"{track_mode}_meta_{timestamp}.txt")
    with open(meta_path, "w", encoding="utf-8") as f:
        for k, v in meta.items():
            f.write(f"{k}: {v}\n")

    print(f"[SNAPSHOT] {track_mode} results kaydedildi:")
    print(f" - {results_path}")
    print(f" - {latest_results_path}")
    print(f" - {meta_path}")


def save_dataset_checkpoint(track_mode, dataset_id, results_list, snapshot_dir=DRIVE_SNAPSHOT_DIR):
    """
    Hem dataset bazlı latest dosyayı, hem de track latest dosyasını Drive'a yazar.
    results_list -> tüm birikmiş sonuçlar
    """
    ensure_snapshot_dir(snapshot_dir)

    if not results_list:
        return

    checkpoint_df = pd.DataFrame(results_list)
    checkpoint_df = normalize_checkpoint_columns(checkpoint_df)

    checkpoint_df = checkpoint_df[checkpoint_df["Track_Mode"] == track_mode].copy()
    if checkpoint_df.empty:
        return

    dataset_df = checkpoint_df[checkpoint_df["Dataset"] == dataset_id].copy()
    if dataset_df.empty:
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    dataset_path = os.path.join(snapshot_dir, f"{track_mode}_dataset_{dataset_id}_{timestamp}.csv")
    latest_dataset_path = os.path.join(snapshot_dir, f"{track_mode}_dataset_{dataset_id}_latest.csv")
    latest_track_path = os.path.join(snapshot_dir, f"{track_mode}_results_latest.csv")

    dataset_df.to_csv(dataset_path, index=False)
    dataset_df.to_csv(latest_dataset_path, index=False)
    checkpoint_df.to_csv(latest_track_path, index=False)

    print(f"[CHECKPOINT] Track={track_mode} Dataset={dataset_id} kaydedildi")
    print(f" - {dataset_path}")
    print(f" - {latest_dataset_path}")
    print(f" - {latest_track_path}")

# Bölüm 9: CheckPoint Handler

In [ ]:
# =========================
# BÖLÜM 9 - LOAD / RESUME
# =========================

def load_track_checkpoint(track_mode, snapshot_dir=DRIVE_SNAPSHOT_DIR):
    """
    Google Drive'daki en güncel track checkpoint'ini yükler.
    Ana kaynak: {track_mode}_results_latest.csv
    """
    ensure_snapshot_dir(snapshot_dir)

    latest_path = os.path.join(snapshot_dir, f"{track_mode}_results_latest.csv")

    if os.path.exists(latest_path):
        try:
            df = pd.read_csv(latest_path)
            df = normalize_checkpoint_columns(df)
            print(f"[RESUME] Drive'dan yüklendi: {latest_path} | satır={len(df)}")
            return df
        except Exception as e:
            print(f"[RESUME] Track checkpoint okunamadı: {e}")

    print(f"[RESUME] Checkpoint bulunamadı: {track_mode}")
    return pd.DataFrame()


def load_dataset_checkpoints(track_mode, snapshot_dir=DRIVE_SNAPSHOT_DIR):
    """
    Track latest yoksa dataset bazlı latest dosyaları Drive'dan toplar.
    """
    ensure_snapshot_dir(snapshot_dir)

    pattern = os.path.join(snapshot_dir, f"{track_mode}_dataset_*_latest.csv")
    files = sorted(glob.glob(pattern))

    if not files:
        print(f"[RESUME] Dataset bazlı checkpoint yok | Track={track_mode}")
        return pd.DataFrame()

    frames = []
    for fp in files:
        try:
            tmp = pd.read_csv(fp)
            tmp = normalize_checkpoint_columns(tmp)
            frames.append(tmp)
        except Exception as e:
            print(f"[RESUME] Okunamadı: {fp} | {e}")

    if not frames:
        return pd.DataFrame()

    merged = pd.concat(frames, ignore_index=True).drop_duplicates()
    merged = normalize_checkpoint_columns(merged)

    print(f"[RESUME] Dataset checkpointlerden toplandı | satır={len(merged)}")
    return merged


def get_completed_runs(checkpoint_df, track_mode, success_only=True):
    """
    Tamamlanmış run seti döner:
    (dataset_id, model_name, seed, track_mode)
    """
    if checkpoint_df is None or checkpoint_df.empty:
        return set()

    df = normalize_checkpoint_columns(checkpoint_df)
    df = df[df["Track_Mode"] == track_mode].copy()

    if df.empty:
        return set()

    if success_only:
        # Hem başarılıları hem de zaman aşımına uğramışları tamamlanmış say
        df = df[df["Status"].isin(["SUCCESS", "TIMEOUT"])].copy()

    completed = set()

    for _, row in df.iterrows():
        if pd.isna(row["Dataset"]) or pd.isna(row["Seed"]) or pd.isna(row["Model"]):
            continue

        completed.add((
            int(row["Dataset"]),
            str(row["Model"]),
            int(row["Seed"]),
            str(row["Track_Mode"])
        ))

    return completed


def bootstrap_resume_state(track_mode, snapshot_dir=DRIVE_SNAPSHOT_DIR):
    """
    Önce track latest dosyasını Drive'dan okumaya çalışır.
    Yoksa dataset latest dosyalarını toplar.
    """
    checkpoint_df = load_track_checkpoint(track_mode, snapshot_dir=snapshot_dir)

    if checkpoint_df.empty:
        checkpoint_df = load_dataset_checkpoints(track_mode, snapshot_dir=snapshot_dir)

        if not checkpoint_df.empty:
            latest_track_path = os.path.join(snapshot_dir, f"{track_mode}_results_latest.csv")
            checkpoint_df.to_csv(latest_track_path, index=False)
            print(f"[RESUME] Track latest yeniden oluşturuldu: {latest_track_path}")

    completed_runs = get_completed_runs(checkpoint_df, track_mode, success_only=True)

    completed_dataset_ids = sorted(list({x[0] for x in completed_runs}))
    print(f"[RESUME] Tamamlanmış run sayısı: {len(completed_runs)}")
    print(f"[RESUME] Başarılı veya Timeout run içeren datasetler: {completed_dataset_ids if completed_dataset_ids else 'Yok'}")

    return checkpoint_df, completed_runs


# Bölüm 10 : Ana Eğitim Döngüsü (Training & Cross-Validation)

In [ ]:
# =========================
# BÖLÜM 10: MAIN LOOP - CV
# =========================


def run_benchmark(track_mode, completed_runs=None, existing_results_df=None):
    global TRACK_MODE

    TRACK_MODE = track_mode

    if completed_runs is None:
        completed_runs = set()

    # Drive'dan okunmuş checkpoint varsa başlangıç state'i olarak kullan
    if existing_results_df is not None and not existing_results_df.empty:
        existing_results_df = normalize_checkpoint_columns(existing_results_df)
        results_list = existing_results_df.to_dict(orient="records")
    else:
        results_list = []

    trained_models = {}

    print(f"\n{'='*60}")
    print(f"RUNNING TRACK: {TRACK_MODE.upper()}")
    print(f"{'='*60}")

    completed_dataset_ids = sorted(list({x[0] for x in completed_runs}))
    print(f"[RESUME] Başarılı tamamlanmış run içeren datasetler: {completed_dataset_ids if completed_dataset_ids else 'Yok'}")
    print(f"[RESUME] Toplam tamamlanmış run sayısı: {len(completed_runs)}")

    # Fazları tanımla: Önce Klasik Modeller, sonra TabNet
    phases = [
        ("Klasik Modeller", lambda m: "TabNet" not in m),
        ("TabNet Modelleri", lambda m: "TabNet" in m)
    ]

    for phase_name, model_filter in phases:
        print(f"\n{'*'*50}")
        print(f" FAZ BAŞLIYOR: {phase_name.upper()}")
        print(f"{'*'*50}")

        for dataset_id in OPENML_DATASET_IDS:
            print(f"\n--- Veri Seti Yükleniyor: OpenML ID {dataset_id} ({phase_name}) ---")

            X_raw, y_raw, metadata = load_openml_data(dataset_id)
            dataset_name = f"{metadata['dataset_name']} | Id: {dataset_id}"
            y_processed = prepare_target(y_raw)

            # EDA'yı sadece ilk fazda yap
            if phase_name == "Klasik Modeller":
                perform_eda(X_raw, y_processed, dataset_id, dataset_name)

            outer_seed = 42
            set_global_seed(outer_seed)

            X_train, X_val, X_test, y_train, y_val, y_test = split_data(
                X_raw,
                y_processed,
                outer_seed
            )

            n_classes = len(np.unique(y_train))

            all_builders = get_model_builders(outer_seed, n_classes)
            filtered_builders = {k: v for k, v in all_builders.items() if model_filter(k)}

            for model_name, model_builder in filtered_builders.items():
                # DÜZELTME: Bütçe ayarlarını dinamik olarak al
                folds, seeds_to_use = get_model_budget(model_name)

                tabnet_override = None
                if "TabNet" in model_name:
                    tabnet_override = get_tabnet_override(dataset_id, TRACK_MODE, model_name)

                if tabnet_override is not None:
                    if tabnet_override.get("skip", False):
                        print(f"Atlandı: {model_name} | Dataset: {dataset_id} | Track: {TRACK_MODE}")
                        continue

                    folds = tabnet_override.get("folds", folds)
                    seeds_to_use = tabnet_override.get("seeds", seeds_to_use)

                class_counts = np.bincount(np.asarray(y_train))
                min_class_count = np.min(class_counts[class_counts > 0])
                folds = max(2, min(folds, min_class_count))

                for seed in SEEDS[:seeds_to_use]:
                    run_key = (dataset_id, model_name, seed, TRACK_MODE)

                    if run_key in completed_runs:
                        print(
                            f"[SKIP] Zaten tamamlanmış | "
                            f"{model_name} | Dataset: {dataset_id} | Seed: {seed} | Track: {TRACK_MODE}"
                        )
                        continue

                    model_time_limit = get_model_time_limit(TRACK_MODE, model_name)
                    model_start_time = time.time()

                    print(
                        f"Eğitiliyor: {model_name} | Dataset: {dataset_id} | "
                        f"Seed: {seed} | Track: {TRACK_MODE} | Limit: {model_time_limit}s"
                    )

                    set_global_seed(seed)

                    timeout_hit = False
                    cv_scores = []

                    skf = StratifiedKFold(
                        n_splits=folds,
                        shuffle=True,
                        random_state=seed
                    )

                    try:
                        for train_idx, fold_val_idx in skf.split(X_train, y_train):
                            if (time.time() - model_start_time) > model_time_limit:
                                timeout_hit = True
                                break

                            X_kf_train = X_train.iloc[train_idx]
                            X_kf_val = X_train.iloc[fold_val_idx]
                            y_kf_train = y_train.iloc[train_idx]
                            y_kf_val = y_train.iloc[fold_val_idx]

                            model = model_builder()

                            fitted_model = fit_model(
                                model_name=model_name,
                                model=model,
                                X_train=X_kf_train,
                                y_train=y_kf_train,
                                X_val=X_val,
                                y_val=y_val,
                                max_epochs=tabnet_override.get("max_epochs") if tabnet_override else None
                            )

                            y_val_pred = fitted_model.predict(X_kf_val)
                            macro_f1 = f1_score(y_kf_val, y_val_pred, average="macro")
                            cv_scores.append(macro_f1)

                        if timeout_hit or len(cv_scores) == 0:
                            elapsed_sec = time.time() - model_start_time
                            print(f"⏱ Timeout | {model_name} | Dataset: {dataset_id} | Seed: {seed}")

                            results_list.append({
                                "Dataset": dataset_id,
                                "Dataset_Name": dataset_name,
                                "Seed": seed,
                                "Model": model_name,
                                "Track_Mode": TRACK_MODE,
                                "Status": "TIMEOUT",
                                "CV_Macro_F1_Mean": np.nan,
                                "CV_Macro_F1_Std": np.nan,
                                "Test_Macro_F1": np.nan,
                                "Primary_Metric_Value": np.nan,
                                "Elapsed_Seconds": elapsed_sec,
                                "Timed_Out": 1,
                            })
                            save_dataset_checkpoint(TRACK_MODE, dataset_id, results_list, DRIVE_SNAPSHOT_DIR)
                            continue

                        cv_mean = np.mean(cv_scores)
                        cv_std = np.std(cv_scores, ddof=1) if len(cv_scores) > 1 else 0.0

                        if (time.time() - model_start_time) > model_time_limit:
                            elapsed_sec = time.time() - model_start_time
                            print(f"⏱ Timeout before final fit | {model_name} | Dataset: {dataset_id} | Seed: {seed}")

                            results_list.append({
                                "Dataset": dataset_id,
                                "Dataset_Name": dataset_name,
                                "Seed": seed,
                                "Model": model_name,
                                "Track_Mode": TRACK_MODE,
                                "Status": "TIMEOUT",
                                "CV_Macro_F1_Mean": np.nan,
                                "CV_Macro_F1_Std": np.nan,
                                "Test_Macro_F1": np.nan,
                                "Primary_Metric_Value": np.nan,
                                "Elapsed_Seconds": elapsed_sec,
                                "Timed_Out": 1,
                            })
                            save_dataset_checkpoint(TRACK_MODE, dataset_id, results_list, DRIVE_SNAPSHOT_DIR)
                            continue

                        final_model = model_builder()

                        fitted_final_model = fit_model(
                            model_name=model_name,
                            model=final_model,
                            X_train=X_train,
                            y_train=y_train,
                            X_val=X_val,
                            y_val=y_val,
                            max_epochs=tabnet_override.get("max_epochs") if tabnet_override else None
                        )

                        y_test_pred = fitted_final_model.predict(X_test)
                        test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")

                        elapsed_sec = time.time() - model_start_time

                        if elapsed_sec > model_time_limit:
                            print(f"⏱ Timeout after final fit | {model_name} | Dataset: {dataset_id} | Seed: {seed}")

                            results_list.append({
                                "Dataset": dataset_id,
                                "Dataset_Name": dataset_name,
                                "Seed": seed,
                                "Model": model_name,
                                "Track_Mode": TRACK_MODE,
                                "Status": "TIMEOUT",
                                "CV_Macro_F1_Mean": np.nan,
                                "CV_Macro_F1_Std": np.nan,
                                "Test_Macro_F1": np.nan,
                                "Primary_Metric_Value": np.nan,
                                "Elapsed_Seconds": elapsed_sec,
                                "Timed_Out": 1,
                            })
                            save_dataset_checkpoint(TRACK_MODE, dataset_id, results_list, DRIVE_SNAPSHOT_DIR)
                            continue

                        results_list.append({
                            "Dataset": dataset_id,
                            "Dataset_Name": dataset_name,
                            "Seed": seed,
                            "Model": model_name,
                            "Track_Mode": TRACK_MODE,
                            "Status": "SUCCESS",
                            "CV_Macro_F1_Mean": cv_mean,
                            "CV_Macro_F1_Std": cv_std,
                            "Test_Macro_F1": test_macro_f1,
                            "Primary_Metric_Value": cv_mean,
                            "Elapsed_Seconds": elapsed_sec,
                            "Timed_Out": 0,
                        })

                        completed_runs.add(run_key)

                        trained_models[(track_mode, dataset_id, seed, model_name)] = {
                            "model": fitted_final_model,
                            "X_test": X_test,
                            "X_train": X_train,
                            "X_val": X_val,
                            "dataset_name": dataset_name,
                            "cv_score": cv_mean,
                            "elapsed_seconds": elapsed_sec,
                        }

                    except Exception as e:
                        elapsed_sec = time.time() - model_start_time
                        print(f"❌ HATA | {model_name} | Dataset: {dataset_id} | Seed: {seed} | {e}")

                        results_list.append({
                            "Dataset": dataset_id,
                            "Dataset_Name": dataset_name,
                            "Seed": seed,
                            "Model": model_name,
                            "Track_Mode": TRACK_MODE,
                            "Status": "FAILED",
                            "Error": str(e),
                            "CV_Macro_F1_Mean": np.nan,
                            "CV_Macro_F1_Std": np.nan,
                            "Test_Macro_F1": np.nan,
                            "Primary_Metric_Value": np.nan,
                            "Elapsed_Seconds": elapsed_sec,
                            "Timed_Out": 1,
                        })

                    # 🔥 HATA ALSA DA BAŞARILI OLSA DA HER SEED SONUNDA KAYDET
                    save_dataset_checkpoint(
                        track_mode=TRACK_MODE,
                        dataset_id=dataset_id,
                        results_list=results_list,
                        snapshot_dir=DRIVE_SNAPSHOT_DIR
                    )

    results_df = pd.DataFrame(results_list)
    results_df = normalize_checkpoint_columns(results_df)

    return results_df, trained_models


# Bölüm 11 : Eğitimleri Başlat

In [ ]:
# =========================
# BÖLÜM 11 : RUN BOTH TRACKS (FINAL)
# =========================

def run_all_tracks(use_checkpoint=True):

    all_results = []
    all_models = {}

    total_start_time = time.time()

    for mode in ["fair", "real"]:

        print(f"\n>>> TRACK BAŞLIYOR: {mode.upper()}")
        print(f"[START] Snapshot Dir: {DRIVE_SNAPSHOT_DIR}")

        if use_checkpoint:
            checkpoint_df, completed_runs = bootstrap_resume_state(
                track_mode=mode,
                snapshot_dir=DRIVE_SNAPSHOT_DIR
            )
        else:
            checkpoint_df = pd.DataFrame()
            completed_runs = set()
            print("[RESUME] Checkpoint kullanımı kapatıldı. Sıfırdan başlanıyor.")

        results_df, trained_models = run_benchmark(
            track_mode=mode,
            completed_runs=completed_runs,
            existing_results_df=checkpoint_df
        )

        save_track_snapshot(
            track_mode=mode,
            results_df=results_df,
            trained_models=trained_models,
            snapshot_dir=DRIVE_SNAPSHOT_DIR
        )

        all_results.append(results_df)

        for k, v in trained_models.items():
            all_models[k] = v

    final_results_df = pd.concat(all_results, ignore_index=True)

    total_minutes = (time.time() - total_start_time) / 60

    merged_path = os.path.join(DRIVE_SNAPSHOT_DIR, "all_tracks_results_latest.csv")
    final_results_df.to_csv(merged_path, index=False)

    print(f"\nTÜM TRACKLER TAMAMLANDI")
    print(f"Toplam süre (tüm benchmark): {total_minutes:.2f} dakika")
    print(f"[SNAPSHOT] birleşik sonuç kaydedildi: {merged_path}")

    return final_results_df, all_models


# =========================
# EXECUTION
# =========================

user_input = input("Önceki checkpoint'ler kullanılsın mı? (E/H): ").strip().lower()
use_checkpoint = True if user_input == 'e' else False

results_df, trained_models = run_all_tracks(use_checkpoint=use_checkpoint)

print("\nBenchmark tamamlandı.")
print(f"Toplam kayıt sayısı: {len(results_df)}")


# Bölüm 12: SHAP - XAI

In [ ]:
def _extract_shap_payload(model, X):

    # CatBoost native
    if isinstance(model, CatBoostNativePipeline):
        X_t = model.transform_features(X)
        feature_names = model.get_feature_names_out().tolist()
        return model.model, X_t, feature_names

    # TabNet
    if isinstance(model, TabNetPipeline):
        X_t = model.transform(X)
        feature_names = [f"f{i}" for i in range(X_t.shape[1])]
        return model.model, X_t, feature_names

    # sklearn pipeline
    if isinstance(model, Pipeline) and "preprocessor" in model.named_steps:
        X_t = model.named_steps["preprocessor"].transform(X)
        feature_names = model.named_steps["preprocessor"].get_feature_names_out().tolist()
        return model.named_steps["model"], X_t, feature_names

    # fallback
    X_np = np.asarray(X)
    feature_names = list(X.columns) if isinstance(X, pd.DataFrame) else [f"f{i}" for i in range(X_np.shape[1])]
    return model, X_np, feature_names

def perform_posthoc_shap(results_df, trained_models, sample_size=500):

    if results_df.empty:
        print("SHAP atlandı: results_df boş")
        return

    print("\n===== SHAP ANALİZİ (POST-HOC) =====")

    # -------------------------
    # TRACK BAZLI AYIR
    # -------------------------
    for track in results_df["Track_Mode"].unique():

        print(f"\n--- TRACK: {track.upper()} ---")

        track_df = results_df[results_df["Track_Mode"] == track]

        # -------------------------
        # BEST MODEL (CV üzerinden!)
        # -------------------------
        best_models_df = (
            track_df
            .groupby(["Dataset", "Model"], as_index=False)["Primary_Metric_Value"]
            .mean()
        )

        best_models_df = best_models_df.loc[
            best_models_df.groupby("Dataset")["Primary_Metric_Value"].idxmax()
        ]

        # -------------------------
        # SHAP LOOP
        # -------------------------
        for dataset_id in best_models_df["Dataset"].unique():

            row = best_models_df[best_models_df["Dataset"] == dataset_id].iloc[0]
            model_name = row["Model"]

            # 🔥 TÜM ADAY MODELLERİ BUL
            candidate_keys = [
                k for k in trained_models
                if k[0] == track and k[1] == dataset_id and k[3] == model_name
            ]

            model = None
            X_test = None
            dataset_name = f"Dataset {dataset_id}"

            if candidate_keys:
                # 🔥 EN İYİ SEED SEÇ (CRITICAL FIX)
                best_key = max(candidate_keys, key=lambda k: trained_models[k]["cv_score"])
                payload = trained_models[best_key]
                model = payload["model"]
                X_test = payload["X_test"]
                dataset_name = payload["dataset_name"]
            else:
                print(f"[{track.upper()}] {model_name} (Dataset: {dataset_id}) hafızada yok. SHAP için sadece bu model yeniden eğitiliyor...")
                try:
                    X_raw, y_raw, metadata = load_openml_data(dataset_id)
                    dataset_name = f"{metadata['dataset_name']} | Id: {dataset_id}"
                    y_processed = prepare_target(y_raw)

                    seed_to_use = 42
                    set_global_seed(seed_to_use)
                    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_raw, y_processed, seed_to_use)

                    n_classes = len(np.unique(y_train))
                    builders = get_model_builders(seed_to_use, n_classes)
                    model_builder = builders[model_name]
                    base_model = model_builder()

                    global TRACK_MODE
                    old_track = TRACK_MODE
                    TRACK_MODE = track

                    tabnet_override = get_tabnet_override(dataset_id, track, model_name) if "TabNet" in model_name else None

                    model = fit_model(
                        model_name=model_name,
                        model=base_model,
                        X_train=X_train,
                        y_train=y_train,
                        X_val=X_val,
                        y_val=y_val,
                        max_epochs=tabnet_override.get("max_epochs") if tabnet_override else None
                    )
                    TRACK_MODE = old_track
                except Exception as e:
                    print(f"Yeniden eğitim başarısız oldu: {e}")
                    continue

            try:
                print(f"\nSHAP → {dataset_name} | Model: {model_name} | Track: {track}")

                # sample (compute control)
                if isinstance(X_test, pd.DataFrame) and len(X_test) > sample_size:
                    X_sample = X_test.sample(sample_size, random_state=42)
                else:
                    X_sample = X_test

                shap_model, shap_input, feature_names = _extract_shap_payload(model, X_sample)

                # sadece tree modeller
                if not hasattr(shap_model, "feature_importances_"):
                    print("SHAP atlandı (tree model değil)")
                    continue

                explainer = shap.TreeExplainer(shap_model)
                shap_values = explainer.shap_values(shap_input)

                if isinstance(shap_values, list):
                    shap_values = shap_values[0]

                plt.figure(figsize=(8, 5))
                # Feature names parametresi shap.summary_plot içerisine eklendi
                shap.summary_plot(shap_values, shap_input, plot_type="bar", show=False, feature_names=feature_names)
                plt.title(f"{dataset_name} | {model_name} | {track}")
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"SHAP başarısız | Dataset: {dataset_name} | Hata: {e}")

# Bölüm 13 : Sonuç Tablosu ve Modellerin Kıyaslanması

In [ ]:
print("\n" + "=" * 50)
print("TÜM MODELLERİN METRİK SONUÇLARI VE KAPSAMLI RAPOR")
print("=" * 50)

pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

results_df = results_df.copy()

# -------------------------
# NUMERIC CAST & EKSİK METRİKLERİN EKLENMESİ
# -------------------------
metric_cols = [
    "Train_Macro_F1",        # Overfitting Gap analizi için gerekli
    "CV_Macro_F1_Mean",
    "CV_Macro_F1_Std",
    "Test_Macro_F1",
    "Log_Loss",              # Olasılık başarı ölçümü için
    "Primary_Metric_Value",
    "Elapsed_Seconds",       # Eğitim süresi
    "Inference_Time"         # Tahmin süresi
]

for col in metric_cols:
    if col in results_df.columns:
        results_df[col] = pd.to_numeric(results_df[col], errors="coerce")

# -------------------------
# SUMMARY & AGGREGATION
# -------------------------
def ci95(x):
    x = pd.Series(x).dropna()
    if len(x) <= 1:
        return np.nan
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x))

# Hangi sütunların var olduğunu kontrol edip dinamik aggregation oluşturuyoruz
agg_dict = {
    "CV_Macro_F1_Mean": ["mean", "std", ci95],
    "Test_Macro_F1": ["mean", "std"],
    "Elapsed_Seconds": ["mean", "std"],
}
if "Log_Loss" in results_df.columns:
    agg_dict["Log_Loss"] = ["mean", "std"]
if "Inference_Time" in results_df.columns:
    agg_dict["Inference_Time"] = ["mean", "std"]

summary_df = results_df.groupby(
    ["Track_Mode", "Dataset", "Dataset_Name", "Model"], as_index=False
).agg(agg_dict)

# Multi-level sütun isimlerini düzleştir
summary_df.columns = ["_".join(col).strip('_') for col in summary_df.columns.values]

# Temel sütun adlarını anlaşılır hale getir
rename_dict = {
    "Track_Mode": "Track_Mode",
    "Dataset": "Dataset",
    "Dataset_Name": "Dataset_Name",
    "Model": "Model",
    "CV_Macro_F1_Mean_mean": "CV_F1_Mean",
    "CV_Macro_F1_Mean_std": "CV_F1_Std",
    "CV_Macro_F1_Mean_ci95": "CV_F1_CI95",
    "Test_Macro_F1_mean": "Test_F1_Mean",
    "Test_Macro_F1_std": "Test_F1_Std",
    "Elapsed_Seconds_mean": "Train_Time_Mean",
    "Elapsed_Seconds_std": "Train_Time_Std"
}
summary_df.rename(columns=rename_dict, inplace=True)

# -------------------------
# OVERFITTING ANALİZİ (TRAIN - CV GAP)
# -------------------------
if "Train_Macro_F1" in results_df.columns and "CV_Macro_F1_Mean" in results_df.columns:
    results_df["Overfit_Gap"] = results_df["Train_Macro_F1"] - results_df["CV_Macro_F1_Mean"]

    plt.figure(figsize=(12, 6))
    sns.barplot(data=results_df, x="Model", y="Overfit_Gap", hue="Track_Mode")
    plt.title("Model Overfitting Analizi (Train-CV Gap)")
    plt.ylabel("F1 Skoru Farkı (Train - Validation)")
    plt.tight_layout()
    plt.show()

# -------------------------
# BOX-PLOT GÖSTERİMİ
# -------------------------
plt.figure(figsize=(14, 7))
sns.boxplot(data=results_df, x="Dataset", y="CV_Macro_F1_Mean", hue="Model")
plt.title("Dataset Bazında Model Performans Dağılımları (Box-Plot)")
plt.ylabel("CV Macro F1 Skor")
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# -------------------------
# FAIR VS REAL DELTA (Δ) METRİĞİ
# -------------------------
print("\n" + "="*50)
print("FAIR vs REAL TUTARLILIĞI VE DELTA (Δ) METRİĞİ")
print("="*50)

pivot_df = summary_df.pivot_table(
    index=["Dataset", "Model"],
    columns="Track_Mode",
    values="CV_F1_Mean"
).reset_index()

if "fair" not in pivot_df.columns:
    pivot_df["fair"] = np.nan
if "real" not in pivot_df.columns:
    pivot_df["real"] = np.nan

pivot_df["Delta (Real - Fair)"] = pivot_df["real"] - pivot_df["fair"]
display(pivot_df.sort_values(by="Delta (Real - Fair)", ascending=False))

# -------------------------
# İSTATİSTİKSEL ANLAMLILIK KONTROLÜ (PAIRED T-TEST)
# -------------------------
print("\n" + "="*50)
print("İSTATİSTİKSEL ANLAMLILIK KONTROLÜ (T-TEST - REAL TRACK)")
print("="*50)

t_test_results = []
datasets = results_df["Dataset"].unique()

for ds in datasets:
    ds_data = results_df[(results_df["Dataset"] == ds) & (results_df["Track_Mode"] == "real")]
    if not ds_data.empty:
        # En iyi modeli seç
        best_model = ds_data.groupby("Model")["CV_Macro_F1_Mean"].mean().idxmax()
        best_scores = ds_data[ds_data["Model"] == best_model]["CV_Macro_F1_Mean"].dropna().values

        for model in ds_data["Model"].unique():
            if model != best_model:
                cmp_scores = ds_data[ds_data["Model"] == model]["CV_Macro_F1_Mean"].dropna().values
                # 2 seed x 5 fold (10) skoru t-test ile karşılaştır
                if len(best_scores) == len(cmp_scores) and len(best_scores) > 1:
                    stat, p_val = ttest_rel(best_scores, cmp_scores)
                    t_test_results.append({
                        "Dataset": ds,
                        "Best_Model": best_model,
                        "Compared_Model": model,
                        "P_Value": round(p_val, 4),
                        "Significant (p<0.05)": p_val < 0.05
                    })

if t_test_results:
    ttest_df = pd.DataFrame(t_test_results)
    display(ttest_df)
else:
    print("T-Test için yeterli eşleşen deneme (seed/fold) bulunamadı.")

# -------------------------
# DATASET BAZLI EN İYİ MODELLER
# -------------------------
summary_df["Rank"] = summary_df.groupby(["Track_Mode", "Dataset"])["CV_F1_Mean"] \
                               .rank(ascending=False, method="dense")

best_models_df = summary_df.loc[
    summary_df.groupby(["Track_Mode", "Dataset"])["CV_F1_Mean"].idxmax()
].reset_index(drop=True)

print("\n" + "="*50)
print("DATASET BAZLI EN İYİ MODELLER")
print("="*50)
display(best_models_df.sort_values(by=["Dataset"]))

# -------------------------
# HİPERPARAMETRE HASSASİYET RAPORU (REAL TRACK)
# -------------------------
print("\n" + "="*50)
print("EN İYİ MODELLERİN HİPERPARAMETRE KONFİGÜRASYONLARI (REAL TRACK)")
print("="*50)

params_list = []
if 'trained_models' in locals() or 'trained_models' in globals():
    for key, data in trained_models.items():
        if key[0] == "real":
            try:
                params = data["model"].get_params()
            except AttributeError:
                params = "Parametreler çekilemedi"

            params_list.append({
                "Dataset": key[1],
                "Model": key[3],
                "Best_Params": str(params)
            })

    if params_list:
        params_df = pd.DataFrame(params_list)
        display(params_df.head()) # Sadece başlıkları gösterir, CSV'ye tümünü atabilirsiniz
else:
    print("trained_models sözlüğü bulunamadı, hiperparametre analizi atlandı.")

# -------------------------
# CONFUSION MATRIX & INFERENCE TIME İÇİN ÖRNEK (BEST MODEL)
# -------------------------
print("\n" + "="*50)
print("EN İYİ MODELLER İÇİN CONFUSION MATRIX & INFERENCE TIME")
print("="*50)

if 'trained_models' in locals() or 'trained_models' in globals():
    for idx, row in best_models_df.iterrows():
        track = row["Track_Mode"]
        d_id = row["Dataset"]
        m_name = row["Model"]

        # İlgili modeli trained_models dict'ten bul
        keys = [k for k in trained_models.keys() if k[0]==track and k[1]==d_id and k[3]==m_name]
        if keys:
            best_key = keys[0]
            model_data = trained_models[best_key]
            model = model_data["model"]
            X_test = model_data["X_test"]
            dataset_name = model_data["dataset_name"]

            try:
                # Geçici veri yükleme ile y_test elde etme (sadece analiz için)
                X_raw, y_raw, _ = load_openml_data(d_id)
                y_processed = prepare_target(y_raw)
                _, _, _, _, _, y_test = split_data(X_raw, y_processed, best_key[2])

                # Inference Time Ölçümü
                start_inf = time.time()
                y_pred = model.predict(X_test)
                inf_time = (time.time() - start_inf)
                avg_inf_time = inf_time / len(X_test) if len(X_test) > 0 else 0

                print(f"[{m_name}] Veri Seti: {d_id} | Toplam Tahmin Süresi: {inf_time:.4f}s | Örnek Başına: {avg_inf_time:.6f}s")

                cm = confusion_matrix(y_test, y_pred)

                plt.figure(figsize=(5, 4))
                sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
                plt.title(f"Confusion Matrix: {m_name}\n(Track: {track}, Dataset: {d_id})")
                plt.xlabel("Tahmin Edilen")
                plt.ylabel("Gerçek")
                plt.show()
            except Exception as e:
                print(f"Confusion matrix/Inference oluşturulamadı: {d_id} - {e}")

# -------------------------
# EXPORT
# -------------------------
summary_df.to_csv("Gazi_ML16_Kapsamli_Ozet_Rapor.csv", index=False)
pivot_df.to_csv("Gazi_ML16_Fair_Real_Delta.csv", index=False)
if t_test_results:
    ttest_df.to_csv("Gazi_ML16_TTest_Sonuclari.csv", index=False)
print("\nRaporlar CSV olarak kaydedildi.")